# Sales Forecasting Model Experiments

This notebook is a presentation-friendly walkthrough of the forecasting engine used by the app.

It focuses on:
- monthly data preparation
- model comparison
- validation metrics
- best-model selection
- one category-level forecast example


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.app_helpers import generate_sample_sales_data, format_inr
from services.forecasting_service import (
    prepare_monthly_series,
    compare_forecast_models,
    forecast_with_best_model,
    forecast_grouped_series,
    get_model_label,
)
from services.evaluation_service import forecast_confidence_score
from services.insight_service import detect_anomalies


## 1. Load Example Sales Data

The app can use uploaded data, but for a repeatable demo this notebook uses the built-in sample dataset.

In [ ]:
df = generate_sample_sales_data()
df["Date"] = pd.to_datetime(df["Date"])
df.head()

In [ ]:
print(f"Rows: {len(df):,}")
print(f"Date range: {df['Date'].min():%d %b %Y} to {df['Date'].max():%d %b %Y}")
print(f"Total sales: {format_inr(df['Sales'].sum())}")
print(f"Categories: {sorted(df['Category'].unique().tolist())}")

## 2. Prepare Monthly Time Series

All forecasting models in the app operate on monthly aggregated sales.

In [ ]:
monthly = prepare_monthly_series(df)
monthly.head()

## 3. Compare Forecast Models

The project evaluates each model on a hold-out test set and ranks them by lowest `MAPE (%)`.

In [ ]:
comparison = compare_forecast_models(monthly)
comparison

In [ ]:
best_row = comparison.iloc[0]
best_model_key = best_row['model_key']
best_model_name = get_model_label(best_model_key)
best_mape = float(best_row['MAPE (%)'])

print(f"Best model: {best_model_name}")
print(f"MAPE: {best_mape:.2f}%")
print(f"Approx. accuracy: {100 - best_mape:.2f}%")

## 4. Explain the Validation Metrics

The app uses these metrics when it judges model quality:

- `MAE`: average absolute error in sales units
- `RMSE`: like MAE, but punishes large misses more strongly
- `MAPE (%)`: average percentage error, easiest to explain in presentations
- `R²`: how much variance the model explains on the hold-out set


## 5. Generate a Forecast with the Best Model

This reproduces the same best-model selection logic used by the app.

In [ ]:
forecast_df, selected_model_key, selected_comparison = forecast_with_best_model(monthly, periods=12)
forecast_df

In [ ]:
forecast_total = forecast_df['Forecast'].sum()
print(f"Selected model: {get_model_label(selected_model_key)}")
print(f"12-month forecast total: {format_inr(forecast_total)}")

## 6. Estimate Forecast Confidence

Confidence in the app is not random. It is based on:
- hold-out forecast error
- sales volatility
- amount of history available
- anomaly count


In [ ]:
anomalies = detect_anomalies(
    monthly.copy(),
    method='rolling_deviation',
    threshold=2.5,
    rolling_window=6,
)
volatility = float(monthly['Sales'].std() / monthly['Sales'].mean() * 100)
confidence = forecast_confidence_score(
    mape=best_mape,
    volatility=volatility,
    data_points=len(monthly),
    anomaly_count=len(anomalies),
)
confidence

## 7. Category-Level Forecast Example

This demonstrates that the grouped forecast logic can generate separate forecasts for each category instead of repeating the same answer.

In [ ]:
group_results = forecast_grouped_series(df, group_col='Category', periods=24, max_groups=5)
group_summary = pd.DataFrame([
    {
        'Category': row['group'],
        'Model': get_model_label(row['model_key']),
        '24-Month Forecast Total': round(row['total_forecast'], 2),
    }
    for row in group_results
]).sort_values('24-Month Forecast Total', ascending=False)
group_summary

In [ ]:
if group_results:
    first_group = group_results[0]
    print('Example category:', first_group['group'])
    print('Model used:', get_model_label(first_group['model_key']))
    display(first_group['forecast_df'].head(6))

## 8. Presentation Notes

Useful summary line for a demo:

- The system compares multiple forecasting models on unseen historical periods.
- It selects the best model by lowest MAPE.
- It can forecast the full business series or separate category-level series.
- Confidence is based on measured error, volatility, history depth, and anomalies.
